In [ ]:
import pandas as pd
import numpy as np 

from mifs import MutualInformationFeatureSelector


https://danielhomola.com/feature%20selection/phd/mifs-parallelized-mutual-information-based-feature-selection-module/#how-to-select-features-using-mutual-information

In [4]:
import sys
sys.path.insert(0, r"C:\Users\jcmar\my_files\SportsBetting\ModelStrategy")
from LogisticRegression.get_train_test import TrainTestBuilder
from feats_config import FeatsConfig

In [5]:
fp = r'C:\Users\jcmar\my_files\SportsBetting\data\training_data\entire_odds_stats_2026-03-09.csv'
df_model = pd.read_csv(fp)

df_model['math_red'] = df_model['math_red'].astype('category')
df_model['math_blue'] = df_model['math_blue'].astype('category')
df_model['elo_pred'] = df_model['elo_pred'].astype('category')


df_model['math_red'] = df_model['math_red'].astype('category')
df_model['math_blue'] = df_model['math_blue'].astype('category')


bad_cols = ['proba_fair_open_red', 'proba_fair_open_blue', 'pimp_open_red', 'pimp_open_blue', 'dec_fair_open_red', 'dec_fair_open_blue',
            'dec_open_red', 'dec_open_blue', 'open_red', 'open_blue', 'juice_open_red', 'juice_open_blue', 'elo_red_proba', 'elo_blue_proba',
            'dog_counts_red', 'dog_counts_blue', 'fav_counts_red', 'fav_counts_blue']

non_diff_feats = [feat for feat in df_model.columns if feat.endswith('_diff') is False]
non_diff_feats = [x for x in non_diff_feats if x not in bad_cols]

y = 'winner'

builder = TrainTestBuilder(df=df_model, target_col=y, non_features=FeatsConfig.non_feats_open, train_size=0.85, random_state=42)
builder.filter_by_date(year=2010, month=2, day=26, date_col='event_date')
X_train, X_test, y_train, y_test, df_train, df_test, scaler_open = builder.prepare_train_test(non_diff_feats, outlier_dict=None, one_hot_encode=True)


Filtered: kept 6650 rows from 2010-02-26 onward.
PREPARE SHAPE: (4256, 297)
MODEL SHAPE: (4177, 297)
Categorical columns: ['weight_class', 'elo_pred', 'math_red', 'math_blue', 'womens_fight']
Numerical columns: ['td_defense_pct_red', 'td_total_attempted_against_red', 'td_total_landed_against_red', 'td_defense_pct_blue', 'td_total_attempted_against_blue', 'td_total_landed_against_blue', 'sig_str_defense_pct_red', 'sig_str_total_attempted_against_red', 'sig_str_total_landed_against_red', 'sig_str_defense_pct_blue', 'sig_str_total_attempted_against_blue', 'sig_str_total_landed_against_blue', 'td_accuracy_pct_red', 'td_accuracy_pct_blue', 'sig_str_accuracy_pct_red', 'sig_str_accuracy_pct_blue', 'kd_pm_red', 'kd_total_red', 'kd_pm_blue', 'kd_total_blue', 'sig_str_landed_pm_red', 'sig_str_landed_total_red', 'sig_str_landed_pm_blue', 'sig_str_landed_total_blue', 'sig_str_absorbed_pm_red', 'sig_str_absorbed_total_red', 'sig_str_absorbed_pm_blue', 'sig_str_absorbed_total_blue', 'sig_str_attempt

In [7]:
# Select numeric columns
numeric_cols = X_train.select_dtypes(include=np.number).columns
variances = X_train[numeric_cols].var()
variance_ranking = variances.sort_values(ascending=False)

print(variance_ranking.head(5))
print(variance_ranking.tail(5))

red_title_fights      1.000282
sub_att_total_blue    1.000282
reverse_total_blue    1.000282
reverse_pm_blue       1.000282
sub_wins_blue         1.000282
dtype: float64
weight_class_Women's Strawweight      0.036332
weight_class_Women's Flyweight        0.031090
weight_class_Women's Bantamweight     0.026318
weight_class_Catch Weight             0.007550
weight_class_Women's Featherweight    0.003650
dtype: float64


In [10]:
X_train.dtypes

td_defense_pct_red                  float64
td_total_attempted_against_red      float64
td_total_landed_against_red         float64
td_defense_pct_blue                 float64
td_total_attempted_against_blue     float64
                                     ...   
weight_class_Women's Strawweight    float64
elo_pred_1                          float64
math_red_1                          float64
math_blue_1                         float64
womens_fight_1                      float64
Length: 169, dtype: object

In [13]:
feat_selector = MutualInformationFeatureSelector(method='MRMR', n_features=35, categorical=False, n_jobs=-1)
feat_selector.fit(X_train, y_train)
X_selected = feat_selector.transform(X_train)



c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but MutualInformationFeatureSelector was fitted without feature names
  warnings.warn(


In [14]:
selected_columns = X_train.columns[feat_selector._support_mask]
print(selected_columns)
print(len(selected_columns))

Index(['td_total_attempted_against_red', 'td_total_landed_against_red',
       'td_total_attempted_against_blue', 'td_total_landed_against_blue',
       'td_landed_total_red', 'td_landed_total_blue', 'td_attempted_total_red',
       'td_attempted_total_blue', 'sub_att_total_red', 'reverse_total_red',
       'reverse_total_blue', 'age_red', 'age_blue', 'reach_red', 'reach_blue',
       'title_fight', 'total_bonus_blue', 'lose_streak_red',
       'lose_streak_blue', 'num_fights_red', 'num_fights_blue', 'num_wins_red',
       'num_wins_blue', 'num_losses_red', 'sub_wins_red', 'sub_wins_blue',
       'decision_losses_red', 'sub_losses_red', 'sub_losses_blue',
       'dec_pct_red', 'ko_pct_blue', 'blue_title_fights', 'red_title_wins',
       'blue_title_wins', 'red_title_losses'],
      dtype='object')
35
